# Chapter 3: Dimensionality Reduction Techniques

**Buku:** scikit-learn Cookbook (O'Reilly)
**Topik:** Teknik reduksi dimensi (PCA, LDA, t-SNE)

## Ringkasan Chapter

**Reduksi dimensi** adalah proses mengurangi jumlah fitur dalam dataset sambil mempertahankan sebanyak mungkin informasi yang relevan. Alasan melakukannya:

- **Menyederhanakan data** agar mudah dianalisis dan divisualisasikan.
- **Menghemat komputasi** (memori dan waktu training).
- **Mengurangi risiko overfitting** dan efek *curse of dimensionality*.

Tiga teknik yang dibahas:

| Teknik | Jenis | Linear? | Tujuan utama |
|:--|:--|:--|:--|
| **PCA** | Unsupervised | Linear | Memaksimalkan variansi yang dipertahankan |
| **LDA** | Supervised | Linear | Memaksimalkan keterpisahan antar kelas |
| **t-SNE** | Unsupervised | Non-linear | Visualisasi 2D/3D dengan menjaga struktur lokal |

> Cara pakai: jalankan sel dari atas ke bawah. Setiap sel kode diikuti penjelasan teori dan cara membaca hasilnya. Bagian t-SNE membutuhkan waktu beberapa puluh detik.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import load_wine, load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split

np.random.seed(2024)
pd.set_option("display.width", 120)

print("scikit-learn:", sklearn.__version__)
print("NumPy       :", np.__version__)
print("pandas      :", pd.__version__)

**Penjelasan:** sel ini memuat seluruh library yang dipakai sepanjang notebook. `np.random.seed(2024)` mengatur biji acak agar hasil bisa direproduksi.

## 1. Mengapa Perlu Reduksi Dimensi?

Ada anggapan bahwa "makin banyak fitur makin bagus", padahal pada dimensi tinggi muncul **curse of dimensionality**: titik-titik data menjadi saling berjauhan dan perbedaan jarak antara tetangga terdekat dan terjauh mengecil, sehingga algoritma berbasis jarak kehilangan daya bedanya.

Kita buktikan dengan eksperimen kecil: ambil 500 titik acak di ruang berdimensi $d$, lalu hitung rasio jarak terdekat terhadap jarak terjauh dari satu titik acuan.

In [ ]:
from sklearn.metrics import pairwise_distances

rng = np.random.RandomState(0)
print("Dimensi | jarak terdekat / jarak terjauh")
for d in [2, 10, 50, 200, 1000]:
    X = rng.uniform(size=(500, d))
    dist = pairwise_distances(X[:1], X[1:]).ravel()
    print(f"{d:>7} | {dist.min() / dist.max():.3f}")

**Penjelasan hasil:** pada 2 dimensi, tetangga terdekat jauh lebih dekat daripada yang terjauh (rasio kecil). Makin besar dimensi, rasio mendekati 1, artinya semua titik hampir sama jaraknya. Konsep "tetangga terdekat" menjadi kurang bermakna. Reduksi dimensi membantu mengatasi masalah ini.

## 2. PCA (*Principal Component Analysis*)

PCA mentransformasi data ke sistem koordinat baru. Sumbu pertama (**PC1**) adalah arah dengan variansi terbesar, sumbu kedua (**PC2**) adalah arah variansi terbesar berikutnya yang **tegak lurus** dengan PC1, dan seterusnya.

Secara matematis, PCA melakukan dekomposisi eigen pada matriks kovarians $\mathbf{C}$:

$$\mathbf{C} = \frac{1}{n-1}\mathbf{X}^T\mathbf{X} = \mathbf{V}\boldsymbol{\Lambda}\mathbf{V}^T$$

Kolom $\mathbf{V}$ adalah arah komponen utama (eigenvector) dan $\lambda_k$ pada diagonal $\boldsymbol{\Lambda}$ adalah variansi yang dijelaskan komponen ke-$k$. Proporsi variansi yang dijelaskan:

$$\text{EVR}_k = \frac{\lambda_k}{\sum_j \lambda_j}$$

### 2.1 Dataset Wine

Dataset ini berisi 178 sampel anggur dari 3 kultivar, masing-masing dengan 13 fitur kimia.

In [ ]:
wine = load_wine()
X_wine, y_wine = wine.data, wine.target
df_wine = pd.DataFrame(X_wine, columns=wine.feature_names)

print("Ukuran data    :", df_wine.shape)
print("Kelas          :", [str(n) for n in wine.target_names])
print("Sampel per kelas:", np.bincount(y_wine))
print()
print(df_wine.head(5).round(2).to_string())
print()
print("Rentang nilai beberapa fitur:")
print(df_wine[["proline", "magnesium", "nonflavanoid_phenols", "hue"]].agg(["min", "max"]).round(2).to_string())

**Penjelasan hasil:** terdapat 178 sampel dengan distribusi kelas 59, 71, dan 48. Perhatikan skala fitur sangat berbeda: `proline` mencapai ratusan hingga ribuan, sedangkan `nonflavanoid_phenols` dan `hue` hanya sekitar 0 sampai 2. PCA mencari arah variansi terbesar, sehingga tanpa standarisasi `proline` akan mendominasi hanya karena angkanya besar. Itu sebabnya **data harus diskalakan sebelum PCA**.

Selain itu, PCA bersifat *unsupervised*: label `y` tidak boleh ikut dalam transformasi. Label hanya dipakai di sini untuk mewarnai plot.

### 2.2 Bukti: PCA tanpa dan dengan scaling

In [ ]:
pca_raw = PCA(n_components=2).fit(X_wine)
pca_std = PCA(n_components=2).fit(StandardScaler().fit_transform(X_wine))

print("Tanpa scaling -> variansi PC1, PC2:", pca_raw.explained_variance_ratio_.round(4))
fitur_dominan = df_wine.columns[np.abs(pca_raw.components_[0]).argmax()]
print("Fitur dengan bobot terbesar di PC1 (tanpa scaling):", fitur_dominan)
print("Bobot fitur tersebut di PC1:", np.abs(pca_raw.components_[0]).max().round(4))
print()
print("Dengan scaling -> variansi PC1, PC2:", pca_std.explained_variance_ratio_.round(4))

**Penjelasan hasil:** tanpa scaling, PC1 praktis hanya merupakan fitur `proline` (bobotnya hampir 1) dan menjelaskan variansi dalam jumlah sangat besar, tetapi ini bukan karena `proline` paling informatif, melainkan karena skalanya besar. Setelah standarisasi, variansi terbagi lebih wajar dan setiap fitur berkontribusi seimbang.

### 2.3 Menerapkan PCA dengan Pipeline

Pipeline memastikan parameter scaling dihitung hanya dari data latih dan diterapkan konsisten, penting untuk mencegah *data leakage* saat cross-validation.

In [ ]:
pca_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2)),
])

X_pca = pca_pipeline.fit_transform(df_wine)

print("Ukuran asli     :", df_wine.shape)
print("Ukuran transform:", X_pca.shape)
print(f"Pengurangan     : {df_wine.shape[1]} fitur -> {X_pca.shape[1]} komponen "
      f"({(1 - X_pca.shape[1]/df_wine.shape[1]):.1%} lebih sedikit)")
print()
print("5 baris pertama (PC1, PC2):")
print(pd.DataFrame(X_pca[:5], columns=["PC1", "PC2"]).round(3).to_string())

**Penjelasan hasil:** 13 fitur kimia dipadatkan menjadi 2 komponen utama (pengurangan sekitar 84.6% jumlah dimensi). Pertanyaan pentingnya: seberapa banyak informasi yang hilang? Kita jawab setelah visualisasi.

### 2.4 Visualisasi hasil PCA

In [ ]:
shapes = ["o", "^", "D"]
colors = ["r", "g", "b"]

pca = pca_pipeline.named_steps["pca"]

plt.figure(figsize=(9, 7))
for i, (shape, color) in enumerate(zip(shapes, colors)):
    plt.scatter(X_pca[y_wine == i, 0], X_pca[y_wine == i, 1],
                c=color, marker=shape, edgecolor="black", alpha=0.8,
                label=wine.target_names[i])

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variansi)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variansi)")
plt.title("Wine Dataset - PCA (2 komponen)")
plt.axhline(0, color="gray", lw=0.5); plt.axvline(0, color="gray", lw=0.5)
plt.legend()
plt.show()

**Penjelasan hasil:** meski PCA tidak pernah melihat label, ketiga kelas anggur sudah terlihat cukup terpisah dalam 2 dimensi, dengan sedikit tumpang tindih antara kelas yang bersebelahan. Itu menandakan variasi terbesar pada data memang berkaitan dengan perbedaan kultivar. Label PC1 dan PC2 mencantumkan persentase variansi yang dijelaskan masing-masing.

### 2.5 Seberapa banyak informasi yang dipertahankan?

*Explained variance ratio* (EVR) menunjukkan proporsi variansi total yang ditangkap tiap komponen.

In [ ]:
evr = pca.explained_variance_ratio_
print(f"PC1: {evr[0]:.4f} ({evr[0]:.2%})")
print(f"PC2: {evr[1]:.4f} ({evr[1]:.2%})")
print(f"Kumulatif 2 komponen: {evr.sum():.4f} ({evr.sum():.2%})")

# Cek: EVR = eigenvalue / total eigenvalue
Xs = StandardScaler().fit_transform(df_wine)
eigvals = np.sort(np.linalg.eigvalsh(np.cov(Xs.T)))[::-1]
print("\nVerifikasi manual dari eigenvalue matriks kovarians:", (eigvals[:2] / eigvals.sum()).round(4))

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["PC1", "PC2"], evr)
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height(), f"{b.get_height():.2%}", ha="center", va="bottom")
ax.set_ylabel("Explained variance ratio")
ax.set_title("Variansi per komponen")
plt.show()

**Penjelasan hasil:** PC1 menangkap sekitar 36.2% dan PC2 sekitar 19.2%, total sekitar 55.4%. Hasil hitungan manual dari eigenvalue matriks kovarians cocok dengan `explained_variance_ratio_`, membuktikan bahwa PCA memang berbasis dekomposisi eigen.

Apakah 55% cukup? Untuk **visualisasi**, cukup baik. Untuk **melatih model**, biasanya dikejar 90-95% variansi.

### 2.6 Scree plot: berapa komponen yang dibutuhkan?

In [ ]:
full_pca = Pipeline([("scaler", StandardScaler()), ("pca", PCA())]).fit(df_wine).named_steps["pca"]
ratios = full_pca.explained_variance_ratio_
cum = np.cumsum(ratios)

print("Komponen | Individual | Kumulatif")
for i, (r, c) in enumerate(zip(ratios, cum), start=1):
    tanda = "  <- pertama kali >= 95%" if c >= 0.95 and (i == 1 or cum[i-2] < 0.95) else ""
    print(f"   PC{i:<2}   |   {r:.4f}   |  {c:.4f}{tanda}")

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(1, len(ratios) + 1)
ax.bar(x, ratios, alpha=0.6, label="Individual")
ax.step(x, cum, where="mid", color="red", lw=2, label="Kumulatif")
ax.axhline(0.95, color="gray", ls="--", label="Ambang 95%")
ax.set_xlabel("Komponen utama"); ax.set_ylabel("Explained variance ratio")
ax.set_xticks(x); ax.legend(); ax.set_title("Scree plot - Wine")
plt.show()

**Penjelasan hasil:** kurva menunjukkan "siku" (*elbow*) setelah 3 komponen pertama: kontribusi turun tajam lalu melandai. Untuk mencapai ambang 95% dibutuhkan **10 dari 13** komponen. Untuk kompresi yang lebih agresif atau sekadar visualisasi, 2-3 komponen sudah memberi gambaran struktur data.

**Cara praktis memilih jumlah komponen:** scikit-learn mengizinkan `n_components` berupa bilangan desimal antara 0 dan 1 yang berarti "pertahankan komponen sebanyak yang diperlukan agar variansi mencapai persentase ini".

In [ ]:
for target in [0.80, 0.90, 0.95]:
    p = Pipeline([("scaler", StandardScaler()), ("pca", PCA(n_components=target))]).fit(df_wine)
    k = p.named_steps["pca"].n_components_
    print(f"Variansi >= {target:.0%} -> {k} komponen")

**Penjelasan hasil:** `PCA(n_components=0.95)` otomatis memilih jumlah komponen yang diperlukan, hasilnya sama dengan pembacaan scree plot. Atribut `n_components_` menyimpan jumlah yang terpilih.

### 2.7 Apa arti komponen utama? (*Loadings*)

Setiap komponen adalah kombinasi linear fitur asli. Bobotnya disimpan di `components_`. Semakin besar nilai absolut bobot, semakin besar kontribusi fitur tersebut pada komponen.

In [ ]:
loadings = pd.DataFrame(pca.components_.T, index=wine.feature_names, columns=["PC1", "PC2"])
print("Fitur dengan kontribusi terbesar:")
print("\nPC1:")
print(loadings["PC1"].reindex(loadings["PC1"].abs().sort_values(ascending=False).index).head(4).round(3).to_string())
print("\nPC2:")
print(loadings["PC2"].reindex(loadings["PC2"].abs().sort_values(ascending=False).index).head(4).round(3).to_string())

# Komponen saling tegak lurus (orthogonal)
print("\nHasil kali dalam PC1 dan PC2 (harus ~0):", np.dot(pca.components_[0], pca.components_[1]).round(10))
print("Panjang vektor PC1 (harus 1):", np.linalg.norm(pca.components_[0]).round(6))

**Penjelasan hasil:** PC1 didominasi oleh fitur-fitur yang berkaitan dengan fenol (misalnya `flavanoids`, `total_phenols`), sedangkan PC2 lebih dipengaruhi fitur seperti `color_intensity` dan `alcohol`. Dengan loadings kita bisa menafsirkan arti tiap sumbu, jadi PCA bukan sepenuhnya "kotak hitam". Hasil kali dalam yang mendekati nol membuktikan bahwa komponen saling tegak lurus.

### 2.8 Berapa banyak yang hilang? Rekonstruksi data

Karena PCA adalah rotasi dan pemotongan dimensi, kita bisa mengembalikan data ke ruang asli dengan `inverse_transform()` dan mengukur *error rekonstruksi*.

In [ ]:
Xs = StandardScaler().fit_transform(df_wine)

print("Komponen | Variansi dipertahankan | MSE rekonstruksi")
for k in [1, 2, 5, 10, 13]:
    p = PCA(n_components=k).fit(Xs)
    rec = p.inverse_transform(p.transform(Xs))
    mse = np.mean((Xs - rec) ** 2)
    print(f"{k:>8} | {p.explained_variance_ratio_.sum():>22.2%} | {mse:.4f}")

**Penjelasan hasil:** semakin banyak komponen dipertahankan, semakin kecil error rekonstruksi. Dengan semua 13 komponen error mendekati 0 karena tidak ada informasi yang dibuang. Perhatikan hubungannya: MSE rekonstruksi per fitur kira-kira sama dengan $1 -$ variansi yang dipertahankan (karena data sudah distandarisasi).

### 2.9 Intuisi geometris: PCA adalah rotasi sumbu

Pada dua fitur saja, PCA hanya memutar sumbu agar sumbu pertama searah dengan penyebaran terbesar. Sebaran titik tidak berubah bentuk.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
pairs = [(0, 1), (6, 8)]
names = wine.feature_names

for col, (f1, f2) in enumerate(pairs):
    Xp = Xs[:, [f1, f2]]
    p2 = PCA(n_components=2).fit(Xp)
    Xr = p2.transform(Xp)

    for i, (m, c) in enumerate(zip(shapes, colors)):
        axes[0, col].scatter(Xp[y_wine == i, 0], Xp[y_wine == i, 1], c=c, marker=m, edgecolor="black", alpha=0.6)
        axes[1, col].scatter(Xr[y_wine == i, 0], Xr[y_wine == i, 1], c=c, marker=m, edgecolor="black", alpha=0.6)

    for k, ac in enumerate(["black", "orange"]):
        v = p2.components_[k] * 3
        axes[0, col].arrow(0, 0, v[0], v[1], color=ac, width=0.03, head_width=0.2)
    axes[0, col].set_title(f"Asli: {names[f1]} vs {names[f2]}")
    axes[0, col].set_xlabel(names[f1]); axes[0, col].set_ylabel(names[f2])
    axes[1, col].set_title("Setelah PCA (rotasi)")
    axes[1, col].set_xlabel("PC1"); axes[1, col].set_ylabel("PC2")

plt.tight_layout()
plt.show()

**Penjelasan hasil:** baris atas menampilkan dua fitur asli beserta panah arah komponen utama (hitam = PC1, oranye = PC2). Baris bawah menampilkan data yang sama setelah sumbu diputar sehingga PC1 mendatar. Bentuk sebaran titik tidak berubah, hanya sudut pandangnya. Itulah inti PCA: mencari sudut pandang yang paling informatif.

## 3. LDA (*Linear Discriminant Analysis*)

LDA juga mencari proyeksi linear, tetapi **memakai label kelas**. Tujuannya memisahkan kelas sejelas mungkin.

PCA memaksimalkan variansi total:

$$\mathbf{w}_{PCA} = \arg\max_{\mathbf{w}}\ \mathbf{w}^T \mathbf{S}_T \mathbf{w}$$

LDA memaksimalkan rasio sebaran **antar kelas** terhadap sebaran **dalam kelas**:

$$\mathbf{w}_{LDA} = \arg\max_{\mathbf{w}}\ \frac{\mathbf{w}^T \mathbf{S}_B \mathbf{w}}{\mathbf{w}^T \mathbf{S}_W \mathbf{w}}$$

dengan $\mathbf{S}_B$ matriks sebaran antar kelas dan $\mathbf{S}_W$ matriks sebaran dalam kelas.

**Batasan:** jumlah komponen LDA maksimal $\min(p, C-1)$, dengan $p$ jumlah fitur dan $C$ jumlah kelas. Pada wine ($C=3$) maksimal 2 komponen.

In [ ]:
lda_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lda", LinearDiscriminantAnalysis(n_components=2)),
])

X_lda = lda_pipeline.fit_transform(X_wine, y_wine)      # LDA membutuhkan y
lda = lda_pipeline.named_steps["lda"]

print("Ukuran asli      :", X_wine.shape)
print("Ukuran transform :", X_lda.shape)
print("Explained variance ratio:", lda.explained_variance_ratio_.round(4))
print("Prior kelas      :", lda.priors_.round(3))
print("Cek proporsi data:", (np.bincount(y_wine) / len(y_wine)).round(3))

**Penjelasan hasil:** 13 fitur dipadatkan menjadi 2 diskriminan linear (LD1 dan LD2). Dua komponen adalah maksimum untuk 3 kelas. Perhatikan: *explained variance ratio* pada LDA mengukur **porsi daya pemisah antar kelas**, bukan variansi total seperti PCA, jadi angkanya tidak bisa dibandingkan langsung. Prior kelas sama dengan proporsi kelas di data (59/178, 71/178, 48/178).

Perbedaan dengan PCA: LDA **wajib diberi `y`** saat `fit`.

In [ ]:
plt.figure(figsize=(9, 7))
for i, (shape, color) in enumerate(zip(shapes, colors)):
    plt.scatter(X_lda[y_wine == i, 0], X_lda[y_wine == i, 1],
                c=color, marker=shape, edgecolor="black", label=wine.target_names[i])
plt.xlabel("LD1"); plt.ylabel("LD2")
plt.title("Wine Dataset - LDA (2 komponen)")
plt.legend()
plt.show()

print("Pusat (centroid) tiap kelas di ruang LDA:")
for i, n in enumerate(wine.target_names):
    print(f"  {n}: {X_lda[y_wine == i].mean(axis=0).round(2)}")

**Penjelasan hasil:** kelompok kelas tampak jauh lebih rapat dan terpisah dibanding PCA. Itu wajar karena LDA secara eksplisit mendorong pusat kelas saling menjauh sambil merapatkan anggota di dalam kelas. Koordinat centroid menunjukkan jarak antar kelas dalam ruang baru.

**Catatan penting:** plot ini memakai LDA yang di-*fit* pada **seluruh** data, jadi tampilannya terlalu optimis sebagai gambaran kemampuan generalisasi. Evaluasi yang adil memakai cross-validation (bagian 6).

### 3.1 PCA vs LDA berdampingan

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, data, title, xl, yl in [
    (axes[0], X_pca, "PCA (unsupervised)", "PC1", "PC2"),
    (axes[1], X_lda, "LDA (supervised)", "LD1", "LD2"),
]:
    for i, (m, c) in enumerate(zip(shapes, colors)):
        ax.scatter(data[y_wine == i, 0], data[y_wine == i, 1], c=c, marker=m, edgecolor="black", label=wine.target_names[i])
    ax.set_title(title); ax.set_xlabel(xl); ax.set_ylabel(yl); ax.legend()
plt.tight_layout()
plt.show()

**Penjelasan hasil:** panel kiri (PCA) memperlihatkan pemisahan yang cukup baik tetapi ada tumpang tindih. Panel kanan (LDA) menunjukkan klaster yang lebih kompak dengan jarak antar klaster lebih lebar.

| Aspek | PCA | LDA |
|:--|:--|:--|
| **Supervisi** | Unsupervised (abaikan label) | Supervised (pakai label) |
| **Tujuan** | Maksimalkan variansi total | Maksimalkan $S_B / S_W$ |
| **Jumlah komponen** | Sampai $p$ | Maksimal $\min(p, C-1)$ |
| **Orthogonalitas** | Komponen saling tegak lurus | Tidak harus tegak lurus |
| **Cocok untuk** | Eksplorasi, kompresi umum | Prapemrosesan klasifikasi |

PCA mencari arah penyebaran terbesar, yang kebetulan sebagian selaras dengan perbedaan kelas. LDA mencari arah yang memang paling membedakan kelas.

## 4. t-SNE (*t-Distributed Stochastic Neighbor Embedding*)

t-SNE adalah teknik **non-linear** yang dirancang khusus untuk **visualisasi** data berdimensi tinggi dalam 2D atau 3D. Cara kerjanya (konseptual):

1. **Ruang asli:** hitung probabilitas $p_{j|i}$ bahwa titik $i$ memilih titik $j$ sebagai tetangga (kernel Gaussian). Titik berdekatan mendapat probabilitas tinggi.
2. **Ruang 2D:** letakkan titik secara acak lalu hitung probabilitas serupa $q_{ij}$ memakai distribusi Student-t (ekor berat).
3. **Optimasi:** geser titik-titik dengan *gradient descent* agar $q$ semirip mungkin dengan $p$, dengan meminimalkan divergensi KL:

$$\text{KL}(P \parallel Q) = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}$$

Hiperparameter terpenting adalah **perplexity** (kira-kira jumlah tetangga efektif tiap titik).

### 4.1 Dataset Digits (64 dimensi)

In [ ]:
digits = load_digits()
Xd, yd = digits.data, digits.target

print("Ukuran data      :", Xd.shape)
print("Ukuran gambar    :", digits.images[0].shape)
print("Jumlah kelas     :", len(np.unique(yd)))
print("Rentang piksel   :", Xd.min(), "-", Xd.max())
print("Sampel per digit :", np.bincount(yd))

fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(digits.images[i], cmap="gray_r")
    ax.set_title(digits.target[i]); ax.axis("off")
plt.show()

**Penjelasan hasil:** terdapat 1.797 gambar tulisan tangan 8x8 piksel (64 fitur), dengan 10 kelas yang hampir seimbang. Ini tugas visualisasi yang lebih sulit daripada wine: 64 dimensi dan 10 kelas. PCA 2 komponen akan kesulitan memisahkan 10 klaster.

### 4.2 PCA vs t-SNE pada Digits

In [ ]:
Xd_s = StandardScaler().fit_transform(Xd)

Xd_pca = PCA(n_components=2).fit_transform(Xd_s)
Xd_tsne = TSNE(n_components=2, random_state=2024).fit_transform(Xd_s)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, data, title in [(axes[0], Xd_pca, "PCA (2 komponen)"), (axes[1], Xd_tsne, "t-SNE (2 komponen)")]:
    sc = ax.scatter(data[:, 0], data[:, 1], c=yd, cmap="tab10", s=12)
    ax.set_title(f"Digits - {title}")
    ax.legend(*sc.legend_elements(), title="Digit", loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

**Penjelasan hasil:** pada PCA (kiri), banyak digit menumpuk dan sulit dibedakan karena proyeksi linear 2D tidak mampu menangkap struktur 64 dimensi. Pada t-SNE (kanan), tiap digit membentuk klaster yang jelas. Artinya data sebenarnya dapat dipisahkan dalam 2D, tetapi lewat pemetaan **non-linear**.

Pola yang wajar dijumpai: digit yang bentuknya mirip (misalnya 3, 5, 8, atau 4 dan 9) cenderung berdekatan, sedangkan angka seperti 0 membentuk klaster yang lebih terpisah.

### 4.3 Pengaruh perplexity

Perplexity mengatur keseimbangan antara struktur lokal dan global. Nilai terlalu kecil menghasilkan banyak klaster kecil yang terpecah, nilai terlalu besar membuat klaster menyatu.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, perp in zip(axes, [5, 30, 100]):
    emb = TSNE(n_components=2, perplexity=perp, random_state=2024).fit_transform(Xd_s)
    ax.scatter(emb[:, 0], emb[:, 1], c=yd, cmap="tab10", s=8)
    ax.set_title(f"perplexity = {perp}")
plt.tight_layout()
plt.show()

**Penjelasan hasil:** bandingkan ketiga panel. Pada perplexity 5 klaster cenderung terpecah menjadi gugus-gugus kecil, pada 30 (default) klaster tampak seimbang, dan pada 100 bentuk global lebih menyatu. Tidak ada nilai yang "benar" secara mutlak, jadi sebaiknya mencoba beberapa nilai (umumnya 5-50).

### 4.4 Keterbatasan t-SNE

Berbeda dengan PCA dan LDA, objek `TSNE` **tidak punya method `transform()`** untuk data baru. Ia hanya punya `fit_transform()`.

In [ ]:
print("TSNE punya transform()? ", hasattr(TSNE(), "transform"))
print("PCA  punya transform()? ", hasattr(PCA(), "transform"))
print("LDA  punya transform()? ", hasattr(LinearDiscriminantAnalysis(), "transform"))

# Dengan inisialisasi acak, random_state berbeda -> koordinat berbeda
a = TSNE(init="random", random_state=1).fit_transform(Xd_s[:500])
b = TSNE(init="random", random_state=2).fit_transform(Xd_s[:500])
print("\nKoordinat titik pertama (seed=1):", a[0].round(2))
print("Koordinat titik pertama (seed=2):", b[0].round(2))

**Penjelasan hasil:** `TSNE` tidak punya `transform()`, sehingga **tidak bisa dipakai di tengah pipeline** untuk memproses data baru. Gunakan hanya untuk eksplorasi dan visualisasi. Selain itu, hasilnya bergantung pada inisialisasi acak (dengan `init="random"` koordinat berbeda di tiap seed; default scikit-learn memakai inisialisasi PCA yang lebih stabil, tetapi hasil optimasi tetap bisa berubah bila parameter lain diganti), dan **posisi absolut maupun jarak antar klaster tidak punya makna tetap**. Yang bermakna hanyalah apakah titik-titik berkelompok atau tidak.

## 5. Memilih Teknik yang Tepat

| Aspek | PCA | LDA | t-SNE |
|:--|:--|:--|:--|
| **Jenis** | Unsupervised | Supervised | Unsupervised |
| **Linearitas** | Linear | Linear | Non-linear |
| **Tujuan** | Maksimalkan variansi | Maksimalkan keterpisahan kelas | Jaga struktur lokal |
| **Dimensi keluaran** | Hingga $\min(n, p)$ | Hingga $\min(p, C-1)$ | Biasanya 2 atau 3 |
| **Bisa transform data baru?** | Ya | Ya | Tidak |
| **Biaya komputasi** | Rendah | Rendah | Tinggi ($\approx O(n^2)$) |
| **Cocok untuk** | Kompresi, ekstraksi fitur | Prapemrosesan klasifikasi | Visualisasi |

**Aturan praktis:** tidak ada label dan ingin kompresi, pakai PCA. Ada label dan tujuannya klasifikasi, pertimbangkan LDA. Ingin melihat struktur klaster, pakai t-SNE (tidak untuk fitur model). Untuk data besar, pola umum adalah PCA dulu (misalnya ke 30-50 komponen) baru t-SNE agar lebih cepat dan lebih bersih dari noise.

In [ ]:
import time

t0 = time.time()
Xd_pca50 = PCA(n_components=30, random_state=0).fit_transform(Xd_s)
emb_a = TSNE(random_state=2024).fit_transform(Xd_pca50)
t_a = time.time() - t0

t0 = time.time()
emb_b = TSNE(random_state=2024).fit_transform(Xd_s)
t_b = time.time() - t0

print(f"PCA(30) lalu t-SNE : {t_a:.1f} detik")
print(f"t-SNE langsung (64): {t_b:.1f} detik")

**Penjelasan hasil:** pada data sekecil digits (64 fitur) waktu kedua cara hampir sama, jadi manfaat kecepatannya belum terlihat. Pola "PCA dulu baru t-SNE" baru terasa pada data dengan ratusan atau ribuan fitur, ketika menghitung jarak di dimensi asli jauh lebih mahal. PCA juga membuang sebagian noise sebelum t-SNE bekerja.

## 6. Dampak Reduksi Dimensi terhadap Performa Model

Reduksi dimensi menyederhanakan model dan mempercepat training, tetapi bisa mengorbankan akurasi bila informasi penting terbuang. Kita ukur secara empiris dengan 5-fold cross-validation. **PCA dan LDA diletakkan di dalam pipeline** sehingga di tiap fold keduanya hanya dipelajari dari data latih fold tersebut.

### 6.1 Wine

In [ ]:
def cv_acc(steps, X, y):
    s = cross_val_score(Pipeline(steps), X, y, cv=5, scoring="accuracy")
    return s.mean(), s.std()

clf = lambda: ("clf", LogisticRegression(max_iter=5000, random_state=2024))

wine_results = {}
wine_results["Semua 13 fitur"] = cv_acc([("sc", StandardScaler()), clf()], X_wine, y_wine)
for n in [2, 5, 8, 10]:
    wine_results[f"PCA-{n}"] = cv_acc([("sc", StandardScaler()), ("pca", PCA(n_components=n)), clf()], X_wine, y_wine)
wine_results["LDA-2"] = cv_acc([("sc", StandardScaler()), ("lda", LinearDiscriminantAnalysis(n_components=2)), clf()], X_wine, y_wine)

print(f"{'Metode':<16} | Akurasi CV (rata-rata +/- std)")
for k, (m, s) in wine_results.items():
    print(f"{k:<16} | {m:.4f} +/- {s:.4f}")

**Penjelasan hasil:** garis dasar (semua fitur) menjadi acuan. PCA dengan 2 komponen biasanya sedikit di bawahnya karena variansi yang terbuang masih berisi informasi untuk klasifikasi. Dengan 5 sampai 10 komponen akurasi mendekati atau menyamai garis dasar. LDA dengan hanya 2 komponen menyamai garis dasar karena langsung dioptimalkan untuk memisahkan kelas. Perbedaan kecil antar angka (sekitar 1 poin) masih berada dalam rentang *std*, jadi jangan terlalu dibaca sebagai perbedaan nyata pada 178 sampel.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
labels = list(wine_results.keys())
vals = [v[0] for v in wine_results.values()]
errs = [v[1] for v in wine_results.values()]
cols = ["darkgreen"] + ["steelblue"] * 4 + ["darkorange"]

bars = ax.bar(labels, vals, yerr=errs, capsize=4, color=cols, edgecolor="black")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.004, f"{v:.2%}", ha="center", fontweight="bold")
ax.axhline(vals[0], color="darkgreen", ls="--", alpha=0.5)
ax.set_ylim(0.85, 1.03)
ax.set_ylabel("Akurasi CV"); ax.set_title("Wine - dampak reduksi dimensi pada akurasi")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

**Penjelasan hasil:** garis putus-putus hijau menandai akurasi garis dasar. Batang diambil langsung dari hasil perhitungan di atas (bukan angka yang ditulis manual), dan garis kecil di atas batang menunjukkan standar deviasi antar fold.

### 6.2 Digits

In [ ]:
digits_results = {}
digits_results["Semua 64 fitur"] = cv_acc([("sc", StandardScaler()), clf()], Xd, yd)
for n in [2, 10, 20, 40]:
    digits_results[f"PCA-{n}"] = cv_acc([("sc", StandardScaler()), ("pca", PCA(n_components=n)), clf()], Xd, yd)
digits_results["LDA-9"] = cv_acc([("sc", StandardScaler()), ("lda", LinearDiscriminantAnalysis(n_components=9)), clf()], Xd, yd)

print(f"{'Metode':<16} | Akurasi CV (rata-rata +/- std)")
for k, (m, s) in digits_results.items():
    print(f"{k:<16} | {m:.4f} +/- {s:.4f}")

**Penjelasan hasil:** pada digits, PCA dengan 2 komponen menurunkan akurasi secara drastis, karena 2 dimensi linear tidak cukup menyimpan informasi 10 kelas (walaupun t-SNE tadi menunjukkan data sebenarnya dapat dipisahkan lewat pemetaan non-linear). Akurasi naik cepat seiring bertambahnya komponen dan mendekati garis dasar sekitar 40 komponen. LDA maksimal menghasilkan $\min(64, 10-1)=9$ komponen, dan pada data ini biasanya memberi akurasi yang bersaing dengan sangat sedikit fitur.

**Pelajaran:** semakin kompleks datanya, semakin banyak komponen yang dibutuhkan. Jumlah komponen sebaiknya dipilih lewat cross-validation, bukan ditebak.

### 6.3 Memilih jumlah komponen PCA dengan GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ("sc", StandardScaler()),
    ("pca", PCA()),
    ("clf", LogisticRegression(max_iter=5000, random_state=2024)),
])

Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.25, random_state=2024, stratify=yd)
grid = GridSearchCV(pipe, {"pca__n_components": [10, 20, 30, 40, 50]}, cv=5)
grid.fit(Xtr, ytr)

print("Jumlah komponen terbaik:", grid.best_params_["pca__n_components"])
print(f"Skor CV terbaik        : {grid.best_score_:.4f}")
print(f"Akurasi data uji       : {grid.score(Xte, yte):.4f}")

**Penjelasan hasil:** `pca__n_components` adalah cara menulis parameter PCA di dalam pipeline. GridSearchCV mencoba beberapa jumlah komponen dan memilih yang skor CV-nya terbaik, lalu skor akhirnya dihitung pada data uji yang belum pernah dipakai. Dengan cara ini jumlah komponen dipilih berdasarkan performa nyata model. Perhatikan bahwa nilai terbaik berada di ujung daftar yang kita coba (50), yang menandakan rentang pencarian sebaiknya diperluas (misalnya sampai 64). Angka di sini juga tidak bisa dibandingkan langsung dengan hasil 5-fold CV sebelumnya karena pembagian datanya berbeda.

## Ringkasan Akhir Chapter 3

| Teknik | Inti | Kelebihan | Keterbatasan |
|:--|:--|:--|:--|
| **PCA** | Rotasi sumbu ke arah variansi terbesar | Cepat, bisa transform data baru, bisa direkonstruksi | Hanya linear, tidak memakai label, wajib scaling |
| **LDA** | Memaksimalkan $S_B / S_W$ | Memakai label, bagus untuk klasifikasi | Maksimal $C-1$ komponen, butuh label |
| **t-SNE** | Menjaga struktur lokal secara non-linear | Visualisasi klaster yang sangat jelas | Tidak ada `transform()`, lambat, hasil acak, jarak antar klaster tidak bermakna |

**Poin penting:**

1. **Standarisasi dulu** sebelum PCA atau LDA, dan letakkan semuanya di dalam `Pipeline`.
2. **Pilih jumlah komponen** lewat scree plot, ambang variansi (misalnya `n_components=0.95`), atau cross-validation.
3. **Pilih teknik sesuai tujuan:** kompresi (PCA), klasifikasi (LDA), visualisasi (t-SNE).
4. **Selalu uji dampaknya pada model.** Pengurangan dimensi yang terlalu agresif menurunkan akurasi, terutama pada data kompleks seperti digits.